# Dunnhumby M5-C: M3 + 사용자별 학습량 보존 M4 (development seed 42)

기존 M4가 고CLV 고객의 **총 학습량**을 키워 저·중CLV 정확도를 깎은 문제만 한 요인으로 수정합니다. 원형 M4와 같은 `q_C × 상품금액백분위 × 고객경제구간적합도`를 쓰되, 각 고객 안에서 중심화·평균 1로 정규화합니다. N·V의 결합인 `q_C`는 그대로 유지됩니다.

- **새 학습 1회:** M3 두 축 그래프 + M4-C
- **재사용:** 기존 M1·M3 seed42 곡선
- **고정:** Dunnhumby 개발 684~690, seed42, 300 epoch, 25 epoch마다 진단; 판정은 300 epoch만
- **주의:** 반복 노출된 개발 구간의 구조 탐색이며 final test·holdout을 생성하거나 평가하지 않습니다.
- **사전 판정:** M5-C의 전체 가격·구매금액 가중 적중값@10·가중 NDCG@10이 M1과 M3보다 모두 높고, 여섯 정확도가 각각 M1·M3의 99% 이상일 때만 후보로 남깁니다.

In [ ]:
from google.colab import drive
from pathlib import Path
import json, os, subprocess, sys

if not os.path.ismount('/content/drive'):
    try:
        drive.mount('/content/drive')
    except (ValueError, NotImplementedError) as exc:
        raise RuntimeError('Drive 연결 실패. 학습을 시작하지 않았습니다.') from exc

SOURCE_COMMIT = '0442600deee652a3ee4327923b1298f72ef67c0f'
REPO = Path('/content/clv-m5-m4c-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', SOURCE_COMMIT], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
os.chdir(REPO)
sys.path.insert(0, str(REPO))

import torch
assert torch.cuda.is_available(), '학습에는 GPU 런타임을 사용하세요.'
import clv_m5_m3_m4_user_centered_screen as screen
screen.self_test()
cfg = screen.configure()
print(json.dumps({'source_commit': SOURCE_COMMIT, 'model': screen.MODEL_ID, 'seed': screen.SEED, 'epochs': screen.FIXED_EPOCH, 'out_dir': cfg.out_dir}, ensure_ascii=False, indent=2))

## 실행

동일 셀을 다시 실행하면 완료 결과는 재사용하고, 중단된 학습은 저장된 checkpoint에서 재개합니다. 새로 학습하는 모형은 M5-C 하나뿐입니다.

In [ ]:
result = screen.run(cfg)

In [ ]:
from IPython.display import display
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

metrics = ['recall@10','ndcg@10','recall@20','ndcg@20','recall@50','ndcg@50',
           'price_purchase_amount_weighted_hit@10','vndcg@10',
           'price_purchase_amount_weighted_hit@20','vndcg@20',
           'price_purchase_amount_weighted_hit@50','vndcg@50']
display(result['absolute'].query('epoch == 300')[['model_id','seed','epoch',*metrics]])
display(result['comparison'].query('epoch == 300 and metric in @metrics'))
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))

zip_path = Path('/content/clv_m5_m3_m4_user_centered_seed42_results.zip')
with ZipFile(zip_path, 'w', ZIP_DEFLATED) as archive:
    for path in result['paths'].values():
        archive.write(path, Path(path).name)
files.download(str(zip_path))